### Google gemini

#### Recommended Tagging Taxonomy

**Tag Key**	Example Values	Purpose

**data_classification** PII, PHI, Public, Internal	Triggers security audits.

**owner_department**	Marketing, Finance, Clinical	Identifies who is responsible for the data.

**refresh_frequency**	Hourly, Daily, Snapshot	Helps users understand data freshness.

**pii_types**	email, ssn, credit_card	Specificity for GDPR/CCPA requests.

**downstream_usage**	PowerBI_Sales_Dashboard	Impact analysis for changes.


In [0]:
# Managing permissions for a specific group
def grant_read_access(catalog_name, principal):
    spark.sql(f"GRANT USAGE ON CATALOG {catalog_name} TO `{principal}`")
    spark.sql(f"GRANT USAGE ON SCHEMA {catalog_name}.default TO `{principal}`")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalog_name}.default TO `{principal}`")
    print(f"Read access granted to {principal}")

grant_read_access("gold_layer", "data_analysts_group")


In [0]:
%sql
-- 1. Create a masking function (SQL)
CREATE OR REPLACE FUNCTION email_mask(email STRING)
  RETURN IF(is_account_group_member('admin'), email, '****@****.com');

-- 2. Apply it to a table (SQL)
ALTER TABLE gold_layer.sales_reports.customers 
ALTER COLUMN email SET MASK email_mask;


In [0]:
# Discovering metadata (Table Properties and Comments)
table_info = spark.sql("DESCRIBE TABLE EXTENDED gold_layer.sales_reports.product_dim")

# Displaying table comments to ensure data discovery documentation exists
display(table_info.filter(table_info.col_name == "Comment"))

# Checking Table Constraints (Important for governance)
constraints = spark.sql("SHOW TBLPROPERTIES gold_layer.sales_reports.product_dim")
display(constraints)


#### 1. Defining Secure Masking Functions

In [0]:
%sql
-- Create a schema for governance logic
CREATE SCHEMA IF NOT EXISTS governance_functions;

-- Email Masking: Only 'hr_admins' see the full email; others see the domain only.
CREATE OR REPLACE FUNCTION governance_functions.email_mask(email STRING)
RETURN IF(is_account_group_member('hr_admins'), email, regexp_replace(email, '^.*@', '****@'));

-- Credit Card Masking: Show only the last 4 digits
CREATE OR REPLACE FUNCTION governance_functions.cc_mask(cc_num STRING)
RETURN IF(is_account_group_member('finance_admins'), cc_num, concat('****-****-****-', right(cc_num, 4)));

-- PHI Redaction: Completely hide SSN/Health IDs for non-medical staff
CREATE OR REPLACE FUNCTION governance_functions.phi_redact(ssn STRING)
RETURN IF(is_account_group_member('medical_staff'), ssn, 'REDACTED');


#### 2. Applying Masks to Tables

In [0]:
%sql
-- Apply masking to a customer PII table
ALTER TABLE main.sensitive_data.customers 
ALTER COLUMN email SET MASK governance_functions.email_mask;

ALTER TABLE main.sensitive_data.customers 
ALTER COLUMN credit_card SET MASK governance_functions.cc_mask;

ALTER TABLE main.sensitive_data.patient_records 
ALTER COLUMN ssn SET MASK governance_functions.phi_redact;


#### 3. Row-Level Security (Data Filtering)

In [0]:
%sql
-- Create a function to filter rows by region
CREATE OR REPLACE FUNCTION governance_functions.region_filter(region STRING)
RETURN is_account_group_member(concat('group_', region)) OR is_account_group_member('admin');

-- Apply the filter to the table
ALTER TABLE main.sensitive_data.patient_records 
SET ROW FILTER governance_functions.region_filter ON (region);


#### 4. Python Script: Identifying & Tagging PII

In [0]:
from pyspark.sql import functions as F

# Sample Data with PII
data = [
    ("John Doe", "john.doe@email.com", "1234-5678-9012-3456", "USA"),
    ("Jane Smith", "jane.s@provider.com", "9876-5432-1098-7654", "UK")
]
df = spark.createDataFrame(data, ["name", "email", "credit_card", "country"])

# Step 1: Write to Unity Catalog
df.write.mode("overwrite").saveAsTable("main.sensitive_data.raw_users")

# Step 2: Use SQL to tag the table with Governance metadata
# This helps data stewards find PII quickly
spark.sql("ALTER TABLE main.sensitive_data.raw_users SET TAGS ('contains_pii' = 'true', 'data_sensitivity' = 'high')")

# Step 3: Verify the tags
display(spark.sql("DESCRIBE TABLE EXTENDED main.sensitive_data.raw_users"))


#### 5. Summary of Best Practices

- Feature	Implementation	Governance Goal
- Column Masking	SET MASK	Protects PII while allowing aggregate analysis.
- Row Filtering	SET ROW FILTER	Restricts data access based on geography or department.
- Tags	SET TAGS	Enables "Data Discovery" and compliance auditing.
- Volumes	GRANT READ	Secures raw unstructured PHI files (scans/images).

#### Script: Applying Multi-Level Tags

In [0]:
# 1. Tagging a Schema (Broad Governance)
spark.sql("""
  ALTER SCHEMA main.healthcare_data 
  SET TAGS ('data_classification' = 'PHI', 'owner' = 'clinical_ops')
""")

# 2. Tagging a Table (Operational Metadata)
spark.sql("""
  ALTER TABLE main.healthcare_data.patient_logs 
  SET TAGS (
    'refresh_frequency' = 'Daily',
    'contains_pii' = 'true',
    'retention_period' = '7_years'
  )
""")

# 3. Tagging a Column (Fine-Grained Discovery)
# Note: Column tags are excellent for identifying specific PII types
spark.sql("""
  ALTER TABLE main.healthcare_data.patient_logs 
  ALTER COLUMN patient_ssn SET TAGS ('pii_type' = 'ssn', 'masking_applied' = 'true')
""")

#### "Security by Design" approach. This script automates the hierarchy: Tagging → Role-Based Access (RBAC) → Redaction (Row Filters) → Masking (Column Filters).

#### Phase 1: Establish the Taxonomy & Masking Functions

In [0]:
%sql
-- Create a centralized governance schema
CREATE CATALOG IF NOT EXISTS governance_root;
CREATE SCHEMA IF NOT EXISTS governance_root.policies;

--- 1. MASKING FUNCTION: Credit Cards (Last 4 digits only) ---
CREATE OR REPLACE FUNCTION governance_root.policies.cc_mask(cc STRING)
RETURN IF(is_account_group_member('finance_admin'), cc, concat('****-****-****-', right(cc, 4)));

--- 2. MASKING FUNCTION: Emails (Obfuscate prefix) ---
CREATE OR REPLACE FUNCTION governance_root.policies.email_mask(email STRING)
RETURN IF(is_account_group_member('data_stewards'), email, regexp_replace(email, '^.*@', '****@'));

--- 3. REDACTION FUNCTION: PII/PHI (Full Redaction for unauthorized) ---
CREATE OR REPLACE FUNCTION governance_root.policies.phi_redact(val STRING)
RETURN IF(is_account_group_member('clinical_specialists'), val, 'REDACTED_SENSITIVE_PHI');

--- 4. ROW FILTER: Regional Governance ---
CREATE OR REPLACE FUNCTION governance_root.policies.region_filter(region STRING)
RETURN is_account_group_member(concat('group_', lower(region))) OR is_account_group_member('admin');


#### Phase 2: Create Principals & Assign Permissions

In [0]:
%sql
-- Note: Groups like 'clinical_specialists' and 'finance_admin' must exist in Account Console
-- We grant 'USAGE' first, which is the 'gatekeeper' permission.

-- Public/General Access
GRANT USAGE ON CATALOG main TO `all_users`;

-- Sensitive Data Access (Restricted to specific principals)
GRANT USAGE ON SCHEMA main.hr_data TO `hr_department_gp`;
GRANT SELECT ON SCHEMA main.hr_data TO `hr_department_gp`;

-- Finance Access
GRANT USAGE ON SCHEMA main.finance_data TO `finance_admin`;
GRANT SELECT ON SCHEMA main.finance_data TO `finance_admin`;


#### Phase 3: Robust Tagging & Policy Application Script

In [0]:
from pyspark.sql import SparkSession

def apply_robust_governance(table_fqn, classification, region_col=None):
    """
    Applies a 3-layer security model to a table.
    1. Layer: Tagging (Discovery)
    2. Layer: Row Filtering (Redaction by Region)
    3. Layer: Column Masking (PII Protection)
    """
    
    print(f"--- Applying Governance to {table_fqn} [{classification}] ---")

    # LAYER 1: TAGGING
    spark.sql(f"ALTER TABLE {table_fqn} SET TAGS ('classification' = '{classification}', 'governance_synced' = 'true')")

    # LAYER 2 & 3: SENSITIVITY-BASED POLICIES
    if classification in ["PHI", "PII_HIGH"]:
        # Apply Row Level Redaction if a region column is provided
        if region_col:
            spark.sql(f"ALTER TABLE {table_fqn} SET ROW FILTER governance_root.policies.region_filter ON ({region_col})")
            print(f"Row filter applied on {region_col}")

        # Identify columns to mask/redact based on standard naming conventions
        cols = spark.table(table_fqn).columns
        
        for col in cols:
            if "email" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK governance_root.policies.email_mask")
            
            elif "credit_card" in col.lower() or "cc_num" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK governance_root.policies.cc_mask")
                
            elif "ssn" in col.lower() or "health_id" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK governance_root.policies.phi_redact")
                
    print(f"Governance sync complete for {table_fqn}.\n")

# --- EXECUTION EXAMPLE ---

# Target Table: main.healthcare.patient_records
# Tag: PHI
# Region-based redaction on column 'data_region'
apply_robust_governance("main.healthcare.patient_records", "PHI", region_col="data_region")

# Target Table: main.finance.transactions
# Tag: PII_HIGH
apply_robust_governance("main.finance.transactions", "PII_HIGH")


### Layered Protection Summary

Tagging (Metadata Layer):

- Action: Sets classification=PHI.
- Effect: Data Stewards can run a query to find all PHI tables across the entire metastore for auditing.

Row Filter (Access Layer):

- Action: SET ROW FILTER region_filter.
- Effect: A user in group_uk will never see rows where region = 'US'. The data is redacted at the query level before it reaches the notebook.

Column Masking (Content Layer):
- Action: SET MASK email_mask.
- Effect: Even if the user is allowed to see the row, the email column will return ****@company.com unless they are in the data_stewards group.

In [0]:
DESCRIBE TABLE EXTENDED main.healthcare.patient_records;